# 502: Decline Statistics by Scenario Type

This notebook calculates decline statistics (delta_warming = peak - 2100) for three scenario groups:
1. NZCO2 scenarios (zeroed after net-zero CO2)
2. NZKyoto scenarios (zeroed after net-zero Kyoto GHG)
3. Base scenarios (no NZCO2 or NZKyoto suffix)

In [1]:
import os
from pathlib import Path

import dotenv
import numpy as np
import pandas as pd

dotenv.load_dotenv()

True

In [2]:
# Configuration
OUTPUT_FOLDER = Path(os.environ["OUTPUT_FOLDER"])

# Input metrics file
MANIFEST_ID = "2026-05-19_09-30"
METRICS_FILE = OUTPUT_FOLDER / f"500_gsat_metrics_{MANIFEST_ID}.csv"

In [3]:
# Load metrics
metrics_df = pd.read_csv(METRICS_FILE)
print(f"Total rows: {len(metrics_df)}")

# Filter for ALL magicc_flag only
metrics_all = metrics_df[metrics_df["magicc_flag"] == "ALL"].copy()
print(f"ALL magicc_flag rows: {len(metrics_all)}")

Total rows: 1839600
ALL magicc_flag rows: 607200


In [4]:
def calculate_scenario_stats(df):
    """
    Calculate per-scenario statistics for delta_warming.
    
    Returns DataFrame with median, 34-67 percentile, and 5-95 percentile per scenario.
    """
    stats = df.groupby(["model", "scenario"]).agg(
        n_ensemble=pd.NamedAgg(column="run_id", aggfunc="count"),
        median=pd.NamedAgg(column="delta_warming", aggfunc="median"),
        p5=pd.NamedAgg(column="delta_warming", aggfunc=lambda x: np.percentile(x, 5)),
        p34=pd.NamedAgg(column="delta_warming", aggfunc=lambda x: np.percentile(x, 34)),
        p67=pd.NamedAgg(column="delta_warming", aggfunc=lambda x: np.percentile(x, 67)),
        p95=pd.NamedAgg(column="delta_warming", aggfunc=lambda x: np.percentile(x, 95)),
    ).reset_index()
    return stats


def calculate_group_summary(scenario_stats):
    """
    Calculate summary statistics across all scenarios in a group.
    
    Returns dict with:
    - median of medians
    - 34-67 percentile range of combined 34-67 values
    - 5-95 percentile range of combined 5-95 values
    """
    return {
        "n_scenarios": len(scenario_stats),
        "median_of_medians": scenario_stats["median"].median(),
        "p34_of_p34": scenario_stats["p34"].quantile(0.34),
        "p67_of_p67": scenario_stats["p67"].quantile(0.67),
        "p5_of_p5": scenario_stats["p5"].quantile(0.05),
        "p95_of_p95": scenario_stats["p95"].quantile(0.95),
    }

## 1. NZCO2 Scenarios

In [5]:
# Filter for NZCO2 scenarios
nzco2_df = metrics_all[metrics_all["scenario"].str.contains("NZCO2", na=False)].copy()
print(f"NZCO2 rows: {len(nzco2_df)}")
print(f"NZCO2 unique scenarios: {nzco2_df['scenario'].nunique()}")

NZCO2 rows: 0
NZCO2 unique scenarios: 0


In [6]:
# Per-scenario statistics
nzco2_stats = calculate_scenario_stats(nzco2_df)
nzco2_stats.head(10)

,model,scenario,n_ensemble,median,p5,p34,p67,p95


In [7]:
# Group summary
nzco2_summary = calculate_group_summary(nzco2_stats)
print("NZCO2 Group Summary:")
for k, v in nzco2_summary.items():
    print(f"  {k}: {v:.3f}" if isinstance(v, float) else f"  {k}: {v}")

NZCO2 Group Summary:
  n_scenarios: 0
  median_of_medians: nan
  p34_of_p34: nan
  p67_of_p67: nan
  p5_of_p5: nan
  p95_of_p95: nan


## 2. NZKyoto Scenarios

In [8]:
# Filter for NZKyoto scenarios
nzkyoto_df = metrics_all[metrics_all["scenario"].str.contains("NZKyoto", na=False)].copy()
print(f"NZKyoto rows: {len(nzkyoto_df)}")
print(f"NZKyoto unique scenarios: {nzkyoto_df['scenario'].nunique()}")

NZKyoto rows: 196800
NZKyoto unique scenarios: 195


In [9]:
# Per-scenario statistics
nzkyoto_stats = calculate_scenario_stats(nzkyoto_df)
nzkyoto_stats.head(10)

,model,scenario,n_ensemble,median,p5,p34,p67,p95
0,AIM/CGE 2.0,SSP1-19_NZKyoto,600,0.2715,0.04990,0.22800,0.31800,0.43505
1,AIM/CGE 2.0,SSP2-19_NZKyoto,600,0.3120,0.11695,0.26600,0.34800,0.45115
2,AIM/CGE 2.1,CD-LINKS-NPi2020_400_NZKyoto,600,0.2550,0.02170,0.20766,0.30000,0.41605
3,AIM/CGE V2.2,ENGAGE-Feasibility-1000/Institutional_NZKyoto,600,0.1620,0.00300,0.12300,0.20500,0.31505
4,AIM/CGE V2.2,ENGAGE-Feasibility-Maximum-Effort/Institutiona...,600,0.2460,0.04790,0.20766,0.29100,0.40410
5,AIM/CGE V2.2,ENGAGE-NPi2020-1000f_NZKyoto,600,0.1025,0.01895,0.08400,0.12900,0.18710
6,AIM/CGE V2.2,ENGAGE-NPi2020-300f_NZKyoto,600,0.2370,0.07590,0.20300,0.27233,0.36505
7,AIM/CGE V2.2,ENGAGE-NPi2020-400f_NZKyoto,600,0.2140,0.06995,0.18400,0.24900,0.34000
8,AIM/CGE V2.2,ENGAGE-NPi2020-500f_NZKyoto,600,0.1860,0.05295,0.16000,0.21700,0.30000
9,AIM/CGE V2.2,ENGAGE-NPi2020-600f_NZKyoto,600,0.1680,0.04000,0.14300,0.19700,0.27410


In [10]:
# Group summary
nzkyoto_summary = calculate_group_summary(nzkyoto_stats)
print("NZKyoto Group Summary:")
for k, v in nzkyoto_summary.items():
    print(f"  {k}: {v:.3f}" if isinstance(v, float) else f"  {k}: {v}")

NZKyoto Group Summary:
  n_scenarios: 328
  median_of_medians: 0.165
  p34_of_p34: 0.096
  p67_of_p67: 0.251
  p5_of_p5: 0.000
  p95_of_p95: 0.457


## 3. Base Scenarios (no NZCO2 or NZKyoto)

In [11]:
# Filter for base scenarios (no NZCO2 or NZKyoto suffix)
base_df = metrics_all[
    ~metrics_all["scenario"].str.contains("NZCO2", na=False) &
    ~metrics_all["scenario"].str.contains("NZKyoto", na=False)
].copy()
print(f"Base scenario rows: {len(base_df)}")
print(f"Base unique scenarios: {base_df['scenario'].nunique()}")

Base scenario rows: 410400
Base unique scenarios: 365


In [12]:
# Per-scenario statistics
base_stats = calculate_scenario_stats(base_df)
base_stats.head(10)

,model,scenario,n_ensemble,median,p5,p34,p67,p95
0,AIM/CGE 2.0,SSP1-19,600,0.2905,0.06595,0.24600,0.33333,0.45005
1,AIM/CGE 2.0,SSP2-19,600,0.3690,0.17900,0.32900,0.40600,0.51605
2,AIM/CGE 2.0,SSP2-26,600,0.0685,0.00295,0.04600,0.09433,0.17710
3,AIM/CGE 2.0,SSP3-34,600,0.0070,0.00000,0.00200,0.01200,0.02800
4,AIM/CGE 2.0,SSP4-26,600,0.0790,0.00800,0.05500,0.09900,0.16005
5,AIM/CGE 2.0,SSP5-26,600,0.0565,0.00000,0.03266,0.08500,0.16915
6,AIM/CGE 2.1,CD-LINKS-NDC2030i_1000,600,0.0000,0.00000,0.00000,0.01133,0.12110
7,AIM/CGE 2.1,CD-LINKS-NPi2020_400,600,0.2650,0.03180,0.21566,0.30800,0.42320
8,AIM/CGE V2.2,ENGAGE-Feasibility-1000/Cost-Effective,600,0.0145,0.00000,0.00000,0.06400,0.16910
9,AIM/CGE V2.2,ENGAGE-Feasibility-1000/Enablers & Institutional,600,0.1430,0.00000,0.10500,0.18233,0.28820


In [13]:
# Group summary
base_summary = calculate_group_summary(base_stats)
print("Base Scenarios Group Summary:")
for k, v in base_summary.items():
    print(f"  {k}: {v:.3f}" if isinstance(v, float) else f"  {k}: {v}")

Base Scenarios Group Summary:
  n_scenarios: 684
  median_of_medians: 0.118
  p34_of_p34: 0.053
  p67_of_p67: 0.215
  p5_of_p5: 0.000
  p95_of_p95: 0.511


## Combined Summary

In [14]:
# Create combined summary table
summary_df = pd.DataFrame([
    {"group": "NZCO2", **nzco2_summary},
    {"group": "NZKyoto", **nzkyoto_summary},
    {"group": "Base", **base_summary},
])

# Round numeric columns
numeric_cols = [c for c in summary_df.columns if c not in ["group", "n_scenarios"]]
summary_df[numeric_cols] = summary_df[numeric_cols].round(3)

summary_df

,group,n_scenarios,median_of_medians,p34_of_p34,p67_of_p67,p5_of_p5,p95_of_p95
0,NZCO2,0,NaN,NaN,NaN,NaN,NaN
1,NZKyoto,328,0.165,0.096,0.251,0.0,0.457
2,Base,684,0.118,0.053,0.215,0.0,0.511


## Save Results

In [15]:
# Save per-scenario statistics
nzco2_stats.to_csv(OUTPUT_FOLDER / f"502_decline_stats_nzco2_{MANIFEST_ID}.csv", index=False)
nzkyoto_stats.to_csv(OUTPUT_FOLDER / f"502_decline_stats_nzkyoto_{MANIFEST_ID}.csv", index=False)
base_stats.to_csv(OUTPUT_FOLDER / f"502_decline_stats_base_{MANIFEST_ID}.csv", index=False)

# Save combined summary
summary_df.to_csv(OUTPUT_FOLDER / f"502_decline_summary_{MANIFEST_ID}.csv", index=False)

print(f"Results saved to {OUTPUT_FOLDER}")

Results saved to /Users/ganti/Documents/code/ar7_netzero_assessment/analysis/output
